# **This notebook performs an additional analysis on validation set and runs inference on model trained with domain adversarial learning**

# **Import libraries**

In [4]:
# Standard Library Imports
import os              # File and directory operations
import math            # Mathematical functions
import shutil          # High-level file operations (copy, move, delete)
import csv             # CSV file reading/writing
from collections import defaultdict

# Data Handling & Processing
import numpy as np         # Numerical computations and arrays
import pandas as pd        # Data manipulation and analysis
import h5py                # HDF5 file handling

# Progress & Visualization
from tqdm import tqdm      # Progress bars for loops
import matplotlib.pyplot as plt  # Plotting and visualization

# PyTorch: Deep Learning
import torch                         # Core PyTorch library
import torch.nn as nn                # Neural network modules
import torch.optim as optim          # Optimizers (Adam, AdamW, etc.)
from torch.utils.data import (       # Dataset and DataLoader utilities
    DataLoader,
    Dataset
)


# **Model architecture**


In [5]:
# Gradient Reversal Layer (GRL)
# Used for domain-adversarial training
class GradReverse(torch.autograd.Function):
    """
    Gradient Reversal Layer (GRL) reverses gradients during backprop.
    This encourages the feature extractor to learn domain-invariant features.
    """
    @staticmethod
    def forward(ctx, x, lambd):
        """
        Forward pass: identity operation.

        Parameters
        ----------
        x : torch.Tensor
            Input features.
        lambd : float
            Gradient reversal coefficient.

        Returns
        -------
        x : torch.Tensor
            Same as input (identity).
        """
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        """
        Backward pass: reverse the gradient and scale by lambda.
        """
        return grad_output.neg() * ctx.lambd, None

def grad_reverse(x, lambd):
    """Convenience function to apply GRL."""
    return GradReverse.apply(x, lambd)

# Token Embedding Module
# Converts raw EEG signals into learned embeddings for LSTM
class TokenEmbedding(nn.Module):
    """
    Convolutional token embedding for EEG channels and temporal sequence.
    """
    def __init__(self, c_in, d_model):
        """
        Parameters
        ----------
        c_in : int
            Number of EEG input channels.
        d_model : int
            Embedding dimension for each time step.
        """
        super(TokenEmbedding, self).__init__()

        # Temporal convolution: expands channels to d_model*4
        self.embed_layer = nn.Sequential(
            nn.Conv2d(1, d_model * 4, kernel_size=(1, 8), padding='same'),
            nn.BatchNorm2d(d_model * 4),
            nn.GELU()
        )

        # Spatial convolution across channels: reduces to d_model
        self.embed_layer2 = nn.Sequential(
            nn.Conv2d(d_model * 4, d_model, kernel_size=(c_in, 1), padding='valid'),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        """
        Forward pass of token embedding.

        Parameters
        ----------
        x : torch.Tensor
            Input EEG tensor of shape (B, C, T)

        Returns
        -------
        x : torch.Tensor
            Embedded tokens of shape (B, T, d_model)
        """
        x = x.unsqueeze(1)            # Add channel dimension: (B, 1, C, T)
        x = self.embed_layer(x)       # Temporal conv
        x = self.embed_layer2(x)      # Spatial conv across channels
        x = x.squeeze(2)              # Remove singleton channel dim
        x = x.permute(0, 2, 1)        # (B, T, d_model) for LSTM
        return x

# DARNet + LSTM (Standard Model)
class DARNet_LSTM(nn.Module):
    """
    DARNet with LSTM for EEG classification (task only, no domain adaptation).
    """
    def __init__(self, c_in=32, d_model=16, hidden=64, num_classes=2):
        super().__init__()
        self.token_embed = TokenEmbedding(c_in, d_model)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden*2, num_classes)  # bidirectional

    def forward(self, x):
        """
        Forward pass.

        Parameters
        ----------
        x : torch.Tensor
            EEG batch (B, C, T)

        Returns
        -------
        out : torch.Tensor
            Task logits (B, num_classes)
        """
        x = self.token_embed(x)        # Token embeddings: (B, T, d_model)
        x, _ = self.lstm(x)            # LSTM output: (B, T, hidden*2)
        x = x.mean(dim=1)              # Global average pooling over time
        out = self.classifier(x)       # Task logits
        return out


# DARNet + LSTM + DANN (Domain-Adversarial)
class DARNet_LSTM_DANN(nn.Module):
    """
    DARNet with LSTM + Domain-Adversarial Neural Network (DANN)
    - Task classifier predicts auditory attention
    - Domain classifier predicts subject ID with GRL
    """
    def __init__(self, c_in=32, d_model=16, hidden=64, num_classes=2, num_subjects=30):
        super().__init__()

        # Token embedding module
        self.token_embed = TokenEmbedding(c_in, d_model)

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Task classifier (main task)
        self.classifier = nn.Linear(hidden*2, num_classes)

        # Domain classifier (subject prediction)
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden*2, 64),
            nn.ReLU(),
            nn.Linear(64, num_subjects)
        )

    def forward(self, x, lambd=0.0):
        """
        Forward pass with optional gradient reversal.

        Parameters
        ----------
        x : torch.Tensor
            EEG batch (B, C, T)
        lambd : float
            Gradient reversal factor (0 during evaluation)

        Returns
        -------
        logits_task : torch.Tensor
            Task prediction logits (B, num_classes)
        logits_domain : torch.Tensor
            Domain (subject) prediction logits (B, num_subjects)
        """
        emb = self.token_embed(x)                  # Token embeddings
        features, _ = self.lstm(emb)              # LSTM output
        feat = features.mean(dim=1)               # Global average pooling

        # Task prediction
        logits_task = self.classifier(feat)

        # Domain prediction with gradient reversal
        rev = grad_reverse(feat, lambd)
        logits_domain = self.domain_classifier(rev)

        return logits_task, logits_domain


# **Load the best trained model**

In [6]:
# Initialize the DARNet LSTM with Domain-Adversarial branch
model = DARNet_LSTM_DANN(d_model=8, num_subjects=30)
# Load pre-trained weights
state_dict = torch.load("/kaggle/input/eeg-aad-track1/best_model.pth")  # Load trained model
model.load_state_dict(state_dict)                                        # Apply weights
# Move model to GPU for faster inference/training
model.to("cuda")

# Initialize the DARNet LSTM without Domain-Adversarial branch
model_lstm = DARNet_LSTM(d_model=8)
# Load pre-trained weights
state_dict_lstm = torch.load("/kaggle/input/darnet-lstm/best_model.pth")  # Load trained model
model_lstm.load_state_dict(state_dict_lstm) # Apply weights
# Move model to GPU for faster inference/training
model_lstm.to("cuda")



DARNet_LSTM(
  (token_embed): TokenEmbedding(
    (embed_layer): Sequential(
      (0): Conv2d(1, 32, kernel_size=(1, 8), stride=(1, 1), padding=same)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
    )
    (embed_layer2): Sequential(
      (0): Conv2d(32, 8, kernel_size=(32, 1), stride=(1, 1), padding=valid)
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
    )
  )
  (lstm): LSTM(8, 64, batch_first=True, bidirectional=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)

# **Calculating extra metric of subject std on validation set, this metric is calculated on both the model with and without domain adversarial learning**

## **Function to load data from h5 file**

In [7]:
def load_h5_dataset_val(file_path):
    """
    Load EEG validation data, labels, and subject IDs from an HDF5 (.h5) file.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 file containing 'data', 'label', and 'sub_id' datasets.

    Returns
    -------
    data : np.ndarray
        EEG data array of shape (N, channels, time_points), e.g., (N, 32, 128).
    label : np.ndarray
        Corresponding labels for each sample.
    subjects : np.ndarray
        1D array of subject IDs associated with each sample.
    """

    # Open the HDF5 file in read-only mode to prevent accidental modifications
    with h5py.File(file_path, 'r') as f:
        data = np.array(f['data'])        # EEG data, shape: (N, 32, 128)
        label = np.array(f['label'])      # Sample labels
        subjects = np.array(f['sub_id'])  # Subject IDs, shape: (N, 1)

    # Squeeze subject array to remove extra dimension
    return data, label, subjects.squeeze()


## **Define custom class for validation data**

In [8]:
class CustomDataset(Dataset):
    """
    PyTorch Dataset for EEG data with labels and subject IDs.

    This class allows seamless integration with PyTorch DataLoader,
    providing batches of (data, label, subject) tensors for training
    and evaluation, including domain-adversarial setups.
    """

    def __init__(self, data, labels, subjects):
        """
        Initialize the dataset.

        Parameters
        ----------
        data : array-like or np.ndarray
            Input EEG data of shape (N, channels, time_points).
        labels : array-like or np.ndarray
            Integer labels for each sample (task targets).
        subjects : array-like or np.ndarray
            Subject IDs corresponding to each sample (used for domain adaptation).
        """
        self.data = data
        self.labels = labels
        self.subjects = subjects

    def __len__(self):
        """
        Return the total number of samples in the dataset.
        """
        return len(self.labels)

    def __getitem__(self, index):
        """
        Retrieve a single sample by index.

        Returns
        -------
        x : torch.FloatTensor
            EEG data for the given sample.
        y : torch.LongTensor
            Label for the given sample.
        s : torch.LongTensor
            Subject ID for the given sample.
        """
        # Convert NumPy arrays to PyTorch tensors
        x = torch.tensor(self.data[index], dtype=torch.float32)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        s = torch.tensor(self.subjects[index], dtype=torch.long)

        return x, y, s


## **Calculated subject standard deviation**

In [10]:
# Paths
val_h5_path = "/kaggle/input/eeg-aad-task1/val_data.h5"  # Validation dataset path

# Load validation data
X_val, y_val, s_val = load_h5_dataset_val(val_h5_path)

# Convert subject IDs to 0-based indexing
unique_subjects = np.unique(s_val)
subject2idx = {sub: i for i, sub in enumerate(unique_subjects)}
s_val = np.array([subject2idx[s] for s in s_val])

# Create DataLoader for validation
val_loader = DataLoader(
    CustomDataset(X_val, y_val, s_val),
    batch_size=128,
    shuffle=False  # Do not shuffle for evaluation
)

# Initialize device and per-subject tracking
device = "cuda"
subject_correct = defaultdict(int)  # Count correct predictions per subject
subject_total = defaultdict(int)    # Count total samples per subject

subject_correct_lstm = defaultdict(int)  # Count correct predictions per subject
subject_total_lstm = defaultdict(int)    # Count total samples per subject

 # Set model to evaluation mode (disables dropout/batchnorm updates)
model.eval()
model_lstm.eval()

# Evaluation loop
with torch.no_grad():  # No gradient computation for efficiency
    for x, y, s in tqdm(val_loader, desc="Evaluating"):
        # Move tensors to device
        x = x.to(device)
        y = y.to(device).long().squeeze(-1)  # Task labels
        s = s.to(device)                      # Subject IDs

        # Forward pass (task prediction only)
        logits, _ = model(x)
        logits_lstm = model_lstm(x)

        # Predicted labels
        preds = torch.argmax(logits, dim=1)
        preds_lstm = torch.argmax(logits_lstm, dim=1)

        # Track per-subject accuracy
        for yi, pi, si in zip(y.cpu(), preds.cpu(), s.cpu()):
            subject_total[int(si)] += 1
            if yi == pi:
                subject_correct[int(si)] += 1

        # Track per-subject accuracy
        for yi, pi, si in zip(y.cpu(), preds_lstm.cpu(), s.cpu()):
            subject_total_lstm[int(si)] += 1
            if yi == pi:
                subject_correct_lstm[int(si)] += 1

# Compute inter-subject standard deviation
# Accuracy for each subject
subject_accs = [subject_correct[sid] / subject_total[sid] for sid in subject_total]
subject_accs_lstm = [subject_correct_lstm[sid] / subject_total_lstm[sid] for sid in subject_total_lstm]
# Standard deviation across subjects — used as a robustness metric
subject_std = np.std(subject_accs)
subject_std_lstm = np.std(subject_accs_lstm)

print(f"Inter-subject STD with DANN(used as val accuracy):  {subject_std:.4f}")
print(f"Inter-subject STD without DANN(used as val accuracy):  {subject_std_lstm:.4f}")
print("==============================\n")


Evaluating: 100%|██████████| 206/206 [00:02<00:00, 74.47it/s]

Inter-subject STD with DANN(used as val accuracy):  0.0380
Inter-subject STD without DANN(used as val accuracy):  0.0371



# **Perform Inference on test data**

## **Function loading data from h5 files**

In [11]:
def load_h5_dataset(file_path):
    """
    Load EEG data and subject IDs from an HDF5 (.h5) file.

    This function is typically used for training datasets where labels
    may not be needed or are handled separately.

    Parameters
    ----------
    file_path : str
        Path to the HDF5 file containing the EEG data.

    Returns
    -------
    data : np.ndarray
        EEG data array of shape (N, channels, time_points), e.g., (N, 32, 128),
        where N is the number of samples.
    subjects : np.ndarray
        1D array of subject IDs corresponding to each sample, squeezed to remove
        any extra dimensions.
    """

    # Open the HDF5 file in read-only mode to safely access datasets
    with h5py.File(file_path, 'r') as f:
        # Load EEG data (shape: N samples × 32 channels × 128 time points)
        data = np.array(f['data'])

        # Load subject IDs (shape: N samples × 1)
        subjects = np.array(f['sub_id'])

    # Remove singleton dimension from subjects and return
    return data, subjects.squeeze()


## **define custom class for test data**

In [12]:
class TestDatasets(Dataset):
    """
    PyTorch Dataset for test EEG data without labels or subject IDs.

    This class is intended for inference/testing only, where we only need
    the input EEG signals to generate predictions.
    """

    def __init__(self, data):
        """
        Initialize the test dataset.

        Parameters
        ----------
        data : array-like or np.ndarray
            Input EEG data of shape (N, channels, time_points), e.g., (N, 32, 128)
        """
        self.data = data

    def __len__(self):
        """
        Return the number of samples in the dataset.
        """
        return len(self.data)

    def __getitem__(self, index):
        """
        Retrieve a single sample as a PyTorch tensor.

        Parameters
        ----------
        index : int
            Index of the sample to retrieve.

        Returns
        -------
        x : torch.FloatTensor
            EEG data for the given sample.
        """
        # Convert NumPy array to PyTorch tensor
        x = torch.tensor(self.data[index], dtype=torch.float32)
        return x


## **Function for running inference**

In [13]:
def run_inference(model, test_loader, device="cuda"):
    """
    Run inference on a test dataset using a trained model.

    Parameters
    ----------
    model : torch.nn.Module
        Trained PyTorch model (e.g., DARNet_LSTM_DANN).
    test_loader : torch.utils.data.DataLoader
        DataLoader for the test dataset.
    device : str, optional
        Device to run inference on ('cuda' or 'cpu'), default is 'cuda'.

    Returns
    -------
    preds : np.ndarray
        Predicted class labels for all test samples.
    """

    model.eval()  # Set model to evaluation mode (disables dropout/batchnorm updates)
    preds = []    # List to store predictions for all batches

    # Disable gradient computation for efficiency
    with torch.no_grad():
        for batch in test_loader:
            # Move batch to the specified device
            batch = batch.to(device)

            # Forward pass through the model
            # Domain classifier branch is disabled during inference
            logits_task, _ = model(batch, lambd=0.0)

            # Convert logits to predicted class labels
            pred = torch.argmax(logits_task, dim=1)

            # Append batch predictions to the list (move to CPU first)
            preds.append(pred.cpu().numpy())

    # Concatenate all batch predictions into a single NumPy array
    preds = np.concatenate(preds, axis=0)
    return preds


## **Function to write predictions in csv file**

In [14]:
def write_submission_csv(preds, sub_id):
    """
    Write predictions to a CSV file for submission.

    Parameters
    ----------
    preds : array-like or np.ndarray
        Predicted labels for the test set.
    sub_id : int or str
        Subject ID used to name the output CSV file.

    Notes
    -----
    - The CSV file will be saved in "/kaggle/working/".
    - Format: two columns ["id", "label"], where "id" is the sample index.
    """

    # Ensure the output directory exists
    os.makedirs("/kaggle/working/", exist_ok=True)

    # Define the CSV file path
    csv_path = f"/kaggle/working/S{sub_id}_pred.csv"

    # Open the file in write mode with UTF-8 encoding
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Write header
        writer.writerow(["id", "label"])

        # Write each prediction along with its sample index
        for i, y in enumerate(preds):
            writer.writerow([i, int(y)])


## **Run the inference**

In [15]:
# List of subject IDs for evaluation or inference
unique_subs = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]

# Run inference for each subject and save predictions to CSV
for ids in tqdm(unique_subs, desc="Subject-wise Inference"):
    
    # Load test data for the current subject
    test_h5_path = f"/kaggle/input/eeg-aad-task1/S{ids}_test_data.h5"
    X_test, _ = load_h5_dataset(test_h5_path)  # Load EEG data (labels not needed)
    
    # Wrap test data into PyTorch Dataset
    test_dataset = TestDatasets(X_test)
    
    # Build DataLoader for batching
    test_loader = DataLoader(
        test_dataset,
        batch_size=128,  # Number of samples per batch
        shuffle=False     # No shuffling for test data
    )

    # Run model inference on the test data
    preds = run_inference(model, test_loader, device="cuda")
    
    # Save predictions to CSV for submission
    write_submission_csv(preds, ids)


Subject-wise Inference: 100%|██████████| 10/10 [00:17<00:00,  1.75s/it]
